<a href="https://colab.research.google.com/github/786RY9/-Graduate-Admission-Prediction-using-ANN/blob/main/Rashid_Yaseen_416636_Lab_7.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lab 7 Prompt Engineering using Google Gemini API

## Install the SDK

The Python SDK for the Gemini API is contained in the [`google-generativeai`](https://pypi.org/project/google-generativeai/) package.

In [2]:
!pip install -q -U google-generativeai

## Set up your API key

To use the Gemini API, you'll need an API key. If you don't already have one, create a key in Google AI Studio.

<a class="button" href="https://aistudio.google.com/app/apikey" target="_blank" rel="noopener noreferrer">Get an API key</a>

In Colab, add the key to the secrets manager under the "🔑" in the left panel. Give it the name `GOOGLE_API_KEY`. Then pass the key to the SDK:

In [3]:
# Import the Python SDK
import google.generativeai as genai
# Used to securely store your API key
from google.colab import userdata

GOOGLE_API_KEY=userdata.get('GOOGLE_API_KEY')
genai.configure(api_key=GOOGLE_API_KEY)

## List models
Use `list_models()` to see what Gemini models are available. These models support `generateContent`, the main method used for prompting.

In [4]:
for m in genai.list_models():
    if "generateContent" in m.supported_generation_methods:
        print(m.name)

models/gemini-2.5-pro-preview-03-25
models/gemini-2.5-flash-preview-05-20
models/gemini-2.5-flash
models/gemini-2.5-flash-lite-preview-06-17
models/gemini-2.5-pro-preview-05-06
models/gemini-2.5-pro-preview-06-05
models/gemini-2.5-pro
models/gemini-2.0-flash-exp
models/gemini-2.0-flash
models/gemini-2.0-flash-001
models/gemini-2.0-flash-exp-image-generation
models/gemini-2.0-flash-lite-001
models/gemini-2.0-flash-lite
models/gemini-2.0-flash-preview-image-generation
models/gemini-2.0-flash-lite-preview-02-05
models/gemini-2.0-flash-lite-preview
models/gemini-2.0-pro-exp
models/gemini-2.0-pro-exp-02-05
models/gemini-exp-1206
models/gemini-2.0-flash-thinking-exp-01-21
models/gemini-2.0-flash-thinking-exp
models/gemini-2.0-flash-thinking-exp-1219
models/gemini-2.5-flash-preview-tts
models/gemini-2.5-pro-preview-tts
models/learnlm-2.0-flash-experimental
models/gemma-3-1b-it
models/gemma-3-4b-it
models/gemma-3-12b-it
models/gemma-3-27b-it
models/gemma-3n-e4b-it
models/gemma-3n-e2b-it
models

## Initialize the Generative Model


In [5]:
model = genai.GenerativeModel('gemini-2.5-pro')

### Generate Content

https://ai.google.dev/api/generate-content#text

In [6]:
response = model.generate_content("Write a story about a magic tree.")
print(response.text)

In the heart of the sleepy village of Oakhaven, nestled in a forgotten grove past the last stone cottage, stood the Whisperwood. It wasn't the oldest or the tallest tree, but it was the most alive. Its bark swirled with iridescent patterns like oil on water, and its leaves were not merely green, but a hundred shades of emerald, jade, and moss, each one shimmering with a faint, internal light.

The children of Oakhaven were told stories about it. They said that if you were quiet enough, you could hear the tree whispering secrets on the wind. Most dismissed it as folklore, the rustling of leaves mistaken for magic. But a young girl named Elara believed.

Elara was a quiet child, with more questions than friends. Her most treasured possession was a small, smooth stone her grandfather had given her before he’d passed away. She often sat by her window, turning the stone over in her palm, feeling the ache of a memory that was beginning to fade around the edges.

One sun-drenched afternoon, d

In [7]:
print(len(response.text))

5682


## Set the Temperature

Every prompt you send to the model includes parameters that control how the model generates responses. Use a `genai.GenerationConfig` to set these, or omit it to use the defaults.

Temperature controls the degree of randomness in token selection. Use higher values for more creative responses, and lower values for more deterministic responses.

You can set the `generation_config` when creating the model.

In [8]:
model = genai.GenerativeModel(
    'gemini-2.5-flash',
    generation_config=genai.GenerationConfig(
        max_output_tokens=2000,   # The maximum number of tokens to include in a response candidate.
        temperature=0.9,
    ))

## Zero-shot Prompting

In [9]:
prompt = "Explain antibiotics"

response = model.generate_content(prompt)

print(response.text)

Antibiotics are a class of powerful medicines used to treat **bacterial infections**. They work by either killing bacteria or slowing their growth, allowing the body's natural immune system to fight off the infection.

Here's a breakdown of what antibiotics are, how they work, their importance, and key considerations:

### What are Antibiotics?

*   **Definition:** Antibiotics are antimicrobial substances that are effective against bacteria. They are ineffective against viruses, fungi, or other pathogens.
*   **Origin:** Many antibiotics were originally discovered in natural sources like fungi (e.g., penicillin from *Penicillium* mold) and bacteria (e.g., streptomycin from *Streptomyces* bacteria). Today, many are semi-synthetic or entirely synthetic.
*   **Discovery:** The first modern antibiotic, penicillin, was discovered by Alexander Fleming in 1928, ushering in a new era of medicine.

### How Do Antibiotics Work? (Mechanisms of Action)

Antibiotics work by targeting specific struc

### Stop Sequences

The set of character sequences (up to 5) that will stop output generation. If specified, the API will stop at the first appearance of a `stop_sequence`. The stop sequence will not be included as part of the response.

In [10]:
prompt = "Give a numbered list of cat facts."

response = model.generate_content(
    prompt,
    # Limit to 5 facts.
    generation_config = genai.GenerationConfig(stop_sequences=['\n5'])
    )

print(response.text)

Here is a numbered list of cat facts:

1.  **Cats can make over 100 different sounds**, whereas dogs can only make about 10.
2.  **A group of cats is called a "clowder."** A male cat is called a "tom" and a female cat is called a "queen."
3.  **Cats spend 70% of their lives sleeping.** That's roughly 13-16 hours a day!
4.  **Domestic cats are believed to have originated from the African Wildcat** and were first domesticated about 9,500 years ago.


### Use system instructions to steer the behavior of a model

System instructions enable you to steer the behavior of the model based on your specific needs and use cases. When you set a system instruction, you give the model additional context to understand the task, provide more customized responses, and adhere to specific guidelines over the full user interaction with the model. You can also specify product-level behavior by setting system instructions, separate from prompts provided by end users.

https://ai.google.dev/gemini-api/docs/system-instructions?lang=python

### Example 1

In [11]:
instruction = "You are a friendly pirate. Speak like one."

model = genai.GenerativeModel(
    "models/gemini-2.5-flash", system_instruction=instruction
)

In [12]:
response = model.generate_content("Good morning! How are you?")
print(response.text)

Ahoy there, matey! A fine mornin' it be, indeed!

Ol' Cap'n [My Name, e.g., Barnacle Bill, or just 'this pirate'] here be shipshape and ready for adventure, thanks for askin'! The winds be fair, the gulls be squawkin', and me timbers be unshi-vered.

And how be ye, me heartie? Ready to seize the day like a chest o' buried treasure?


### Example 2

In [13]:
error_handling_system_prompt =f"""
Your task is to explain exactly why this error occurred and how to fix it.
"""
error_handling_model = genai.GenerativeModel(model_name='gemini-2.5-flash', generation_config={"temperature": 0},
                                             system_instruction=error_handling_system_prompt)

In [14]:
from IPython.display import Markdown

error_message = """
   1 my_list = [1,2,3]
----> 2 print(my_list[3])

IndexError: list index out of range
"""

error_prompt = f"""
You've encountered the following error message:
Error Message: {error_message}"""

Markdown(error_handling_model.generate_content(error_prompt).text)

This `IndexError: list index out of range` occurs because you are trying to access an element in the list using an index that does not exist.

Let's break it down:

**Why the Error Occurred:**

1.  **Zero-Based Indexing:** In Python (and many other programming languages), list indices start at `0`, not `1`.
    *   For `my_list = [1, 2, 3]`:
        *   The element `1` is at index `0`.
        *   The element `2` is at index `1`.
        *   The element `3` is at index `2`.
2.  **List Length:** Your list `my_list` has 3 elements.
3.  **Invalid Index:** The valid indices for `my_list` are `0`, `1`, and `2`. When you try to access `my_list[3]`, you are asking for the *fourth* element, which does not exist in a list that only has three elements. The index `3` is "out of range."

**How to Fix It:**

You need to use a valid index (0, 1, or 2) to access an existing element, or add more elements to the list if you intend to access a position beyond its current length.

Here are a few common scenarios and their fixes:

**1. If you wanted to access an existing element (e.g., the last element):**

Change the index to a valid one. If you wanted the last element, use index `2` (or `len(my_list) - 1`, or `-1` for convenience).

```python
my_list = [1, 2, 3]

# To access the first element (value 1):
print(my_list[0]) # Output: 1

# To access the second element (value 2):
print(my_list[1]) # Output: 2

# To access the third (and last) element (value 3):
print(my_list[2]) # Output: 3

# A common way to get the last element is using negative indexing:
print(my_list[-1]) # Output: 3
```

**2. If you wanted to add a new element to the list:**

You cannot add an element by simply assigning to an out-of-range index like `my_list[3] = 4`. You need to use methods like `append()` or `insert()`.

```python
my_list = [1, 2, 3]

# To add an element to the end of the list:
my_list.append(4)
print(my_list) # Output: [1, 2, 3, 4]
print(my_list[3]) # Now this is valid and outputs: 4

# To insert an element at a specific position:
my_list.insert(0, 0) # Inserts 0 at index 0
print(my_list) # Output: [0, 1, 2, 3, 4]
```

**3. If you are iterating or dealing with dynamic indices:**

Always ensure your index is within the valid range of the list's length.

```python
my_list = [1, 2, 3]
index_to_access = 3 # This would cause the error

if index_to_access < len(my_list):
    print(my_list[index_to_access])
else:
    print(f"Error: Index {index_to_access} is out of range for list of length {len(my_list)}")
    # Or handle the case, e.g., append a new element, or use a default value
```

### Example 3

In [15]:
code_generation_system_prompt = f"""
You are a coding assistant. Your task is to generate a code snippet that accomplishes a specific goal.
The code snippet must be concise, efficient, and well-commented for clarity.
Consider any constraints or requirements provided for the task.

If the task does not specify a programming language, default to Python.
"""
code_generation_model = genai.GenerativeModel(model_name='gemini-2.5-flash', generation_config={"temperature": 0},
                                             system_instruction=code_generation_system_prompt)

In [16]:
code_generation_prompt = 'Create a countdown timer that ticks down every second and prints "Time is up!" after 20 seconds'

Markdown(code_generation_model.generate_content(code_generation_prompt).text)

```python
import time

def countdown_timer(seconds):
    """
    A simple countdown timer that ticks down every second.

    Args:
        seconds (int): The initial number of seconds to count down from.
    """
    print(f"Starting countdown from {seconds} seconds...")
    for i in range(seconds, 0, -1):
        print(f"{i} seconds remaining...")
        time.sleep(1)  # Pause for 1 second
    print("Time is up!")

if __name__ == "__main__":
    countdown_timer(20)
```

In [17]:
import time

# Set the countdown duration in seconds
countdown_duration = 20

# Start the countdown
for i in range(countdown_duration, 0, -1):
    print(i, end=" ")
    time.sleep(1)  # Wait for 1 second

# Print "Time is up!" after the countdown
print("Time is up!")

20 19 18 17 16 15 14 13 12 11 10 9 8 7 6 5 4 3 2 1 Time is up!


## Few-shot Prompting

### Example 1

In [18]:
prompt = """
A "whatpu" is a small, furry animal native to Tanzania. An example of a sentence that uses the word whatpu is:
We were traveling in Africa and we saw these very cute whatpus.

To do a "farduddle" means to jump up and down really fast. An example of a sentence that uses the word farduddle is:
"""

response = model.generate_content(prompt)
print(response.text)

Ahoy there, matey! A fine set o' words ye be sharin'!

To do a "farduddle" means to jump up and down really fast. An example of a sentence that uses the word farduddle is:

"When the captain spotted land after weeks at sea, he began to farduddle with such excitement, he nearly knocked over the ship's parrot!"

Yarrr! Hope that helps ye on yer linguistic voyage!


### Example 2

In [19]:
prompt = """
This is awesome! // Negative
This is bad! // Positive
Wow that movie was rad! // Positive
What a horrible show! //
"""

model2 = genai.GenerativeModel('gemini-2.5-flash')

response = model2.generate_content(prompt)
print(response.text)

What a horrible show! // Positive


## Chain-of-Thought (CoT) Prompting

In [20]:
prompt = """
The odd numbers in this group add up to an even number: 4, 8, 9, 15, 12, 2, 1.
A: Adding all the odd numbers (9, 15, 1) gives 25. The answer is False.

The odd numbers in this group add up to an even number: 17,  10, 19, 4, 8, 12, 24.
A: Adding all the odd numbers (17, 19) gives 36. The answer is True.

The odd numbers in this group add up to an even number: 16,  11, 14, 4, 8, 13, 24.
A: Adding all the odd numbers (11, 13) gives 24. The answer is True.

The odd numbers in this group add up to an even number: 17,  9, 10, 12, 13, 4, 2.
A: Adding all the odd numbers (17, 9, 13) gives 39. The answer is False.

The odd numbers in this group add up to an even number: 15, 32, 5, 13, 82, 7, 1.
A:
"""

response = model2.generate_content(prompt)
print(response.text)

Adding all the odd numbers (15, 5, 13, 7, 1) gives 41. The answer is False.


## Let's think Step-by-Step

Let's try a simple problem and see how the model performs:

In [21]:
prompt = """
I went to the market and bought 10 apples. I gave 2 apples to the neighbor and 2 to the repairman. I then went and bought 5 more apples and ate 1. How many apples did I remain with?
"""

response = model2.generate_content(prompt)
print(response.text)

Let's break it down step-by-step:

1.  **Started with:** 10 apples
2.  **Gave to neighbor:** 10 - 2 = 8 apples
3.  **Gave to repairman:** 8 - 2 = 6 apples
4.  **Bought more:** 6 + 5 = 11 apples
5.  **Ate one:** 11 - 1 = 10 apples

You remained with **10 apples**.


In [22]:
prompt = """
I went to the market and bought 10 apples. I gave 2 apples to the neighbor and 2 to the repairman. I then went and bought 5 more apples and ate 1. How many apples did I remain with?

Let's think step by step.
"""

response = model2.generate_content(prompt)
print(response.text)

Let's break it down step by step:

1.  **Started with:** 10 apples
2.  **Gave to neighbor:** 10 - 2 = 8 apples
3.  **Gave to repairman:** 8 - 2 = 6 apples
4.  **Bought more:** 6 + 5 = 11 apples
5.  **Ate one:** 11 - 1 = 10 apples

You remained with **10 apples**.


## Tasks

write prompts for the following



### Simple Prompting Situations
- Write a thank-you note to a teacher.
- Write a short poem about the moon.
- Describe what a volcano is in one sentence.
- Create a motivational quote.


In [24]:
instructions = {"Write a polite thank-you message to a school teacher.", "Write a short poem about the moon.", "Describe what a volcano is in one sentence.","Create a motivational quote."}

model = genai.GenerativeModel(
    'gemini-2.5-flash',
    generation_config=genai.GenerationConfig(
        max_output_tokens=2000,
        temperature=0.8,
    ))
for instruction in instructions:

  response = model.generate_content(instructions)
  # print(response)
  print(instruction)
  print(response.text)

Write a polite thank-you message to a school teacher.
Here are your requested items:

**Polite Thank-You Message to a School Teacher:**

Dear [Teacher's Name],

I hope this message finds you well.

I wanted to express my sincere gratitude for your dedication and effort this past [term/year/during X project]. Your [specific quality, e.g., patience, engaging lessons, helpful guidance] truly made a positive difference in [my/my child's] learning experience.

Thank you for everything you do.

Sincerely,
[Your Name/Student's Name/Parent's Name]

---

**Short Poem About the Moon:**

Silver disc in velvet night,
A gentle, soft, and silent light.
It watches as the world sleeps deep,
Secrets that the shadows keep.

---

**Motivational Quote:**

"Your potential is limitless; let your courage be the spark that ignites it."

---

**Description of a Volcano in One Sentence:**

A volcano is a vent in the Earth's crust through which molten rock, ash, and gases erupt.
Write a short poem about the moon

### Zero-Shot Prompts situations
- Explain what blockchain is to a 10-year-old.
- Predict the next line of a conversation between two astronauts.
- Identify the tone of a paragraph (humorous, serious, etc.).
- Write a short scene about two characters arguing in a coffee shop.

In [25]:
prompts = [
    # 1) Explain blockchain to a 10-year-old
    "Explain what blockchain is in simple words that a 10-year-old can easily understand. Use a friendly tone and avoid technical terms.",

    # 2) Predict next line of conversation between two astronauts
    "Two astronauts are talking while floating in space. Predict the next line in their conversation based on context and a realistic tone.",

    # 3) Identify the tone of a paragraph
    "Read the following paragraph and identify the tone (e.g., humorous, serious, sarcastic, formal, excited, etc.), then briefly justify your answer.\nParagraph: <INSERT_PARAGRAPH_HERE>",

    # 4) Write a short scene of an argument in a coffee shop
    "Write a short scene where two characters are arguing in a coffee shop. Include dialogue and describe the emotions and atmosphere."
]

for p in prompts:
    response = model.generate_content(p)
    print("\nPrompt:", p)
    if response.text:
        print("Response:", response.text)
    else:
        print("Response not generated (safety filter or token issue).")



Prompt: Explain what blockchain is in simple words that a 10-year-old can easily understand. Use a friendly tone and avoid technical terms.
Response: Hey there, super smart kid!

Imagine you and your friends are playing a really fun game, and you need to keep track of important things, like who traded a shiny sticker for a cool toy, or who scored points.

Instead of just one person writing it all down in *their* notebook (which they could secretly change!), you decide to do something much cooler:

1.  **The Storybook:** You create a super special, super long storybook. This book is for *everyone* playing the game.

2.  **Pages of Events (Blocks):** Whenever something important happens – like "Sarah traded her blue sticker for Tom's red toy" – you write it down on a page in the storybook. Each page is like a "block" of information. When a page gets full, you start a new one.

3.  **Linking the Pages (Chain):** Here's the magic part! You don't just put the pages anywhere. Each new page 

### Few-Shot Prompts situations
- Translate idioms from English to Spanish with explanations.
- Classify customer feedback as Positive, Negative, or Neutral.
- Rewrite formal sentences into casual conversation.
- Provide critiques on writing samples using 2 examples of feedback first.

In [26]:
prompts = [

    # 1) Translate idioms EN → ES with explanation
    """You are an expert translator who preserves idiomatic meaning, not literal translation.

Example:
English: "Break a leg"
Spanish: "¡Mucha suerte!"
Explanation: This idiom is used to wish someone good luck.

Example:
English: "Piece of cake"
Spanish: "Muy fácil"
Explanation: Used to describe something that is very easy.

Now translate the following idiom and explain it:
English: "How Lucky are you?"
""",

    # 2) Classify customer feedback tone
    """You are a sentiment classifier.

Example:
Feedback: "The product works great, I'm very satisfied."
Classification: Positive

Example:
Feedback: "It arrived broken and nobody is helping me fix it."
Classification: Negative

Example:
Feedback: "It's okay, not good, not bad."
Classification: Neutral

Now classify the following customer feedback:
"It was a good Session and activities were fun."
""",

    # 3) Rewrite formal → casual
    """Rewrite formal sentences into friendly, casual conversation.

Example:
Formal: "I would appreciate it if you could provide the report by tomorrow."
Casual: "Hey, could you send the report by tomorrow? Thanks!"

Example:
Formal: "I will not be able to attend the meeting due to prior commitments."
Casual: "I can't make it to the meeting, I already have something planned."

Now rewrite:
Formal: "I am looking forward to seeing you tomorrow at the conference."
""",

    # 4) Critique writing with two example structures
    """You are a supportive writing critic. Always provide:
1. One positive aspect (Strength)
2. One clear improvement suggestion (Suggestion)

Example:
Writing: "The sun set over the horizon as I walked home."
Feedback:
- Strength: Great visual imagery.
- Suggestion: Add emotion to enhance the scene's mood.

Example:
Writing: "The presentation was given by the manager."
Feedback:
- Strength: The meaning is clear.
- Suggestion: Try using active voice for more engagement.

Now critique the following writing:
"To walk in nature is as witenessing thousand miracles."
"""
]

for p in prompts:
    response = model.generate_content(p)
    print("\nPrompt:\n", p)
    if response.candidates and response.candidates[0].content.parts:
        print("Response:\n", response.candidates[0].content.parts[0].text)
    else:
        print("No text returned (safety or token issue).")



Prompt:
 You are an expert translator who preserves idiomatic meaning, not literal translation.

Example:
English: "Break a leg"
Spanish: "¡Mucha suerte!"
Explanation: This idiom is used to wish someone good luck.

Example:
English: "Piece of cake"
Spanish: "Muy fácil"
Explanation: Used to describe something that is very easy.

Now translate the following idiom and explain it:
English: "How Lucky are you?"

Response:
 **English:** "How Lucky are you?"
**Spanish:** "¡Qué suerte tienes!"

**Explanation:** This phrase is used to comment on someone's fortune. It can be said with genuine surprise or admiration when someone experiences good luck, or it can be used ironically and sarcastically when someone is in an unfortunate situation, implying they are quite unlucky. The intended meaning is often conveyed through the speaker's tone of voice.

Prompt:
 You are a sentiment classifier.

Example:
Feedback: "The product works great, I'm very satisfied."
Classification: Positive

Example:
Feedb


### Chain-of-Thought (CoT) Prompts situations
- Solve a math word problem (e.g., age-based puzzles).
- Analyze if a statement is logically valid or not.
- Determine if a story contains a plot hole.
- Recommend the best laptop for a student with a $500 budget and justify the choic

In [27]:
prompts = {

    # 1) Math word problem (age puzzle)
    "math_age": """
Solve each age problem. For each problem:
A: Start with "Answer:" followed by ages (e.g., "Answer: Alice = 14, Bob = 7").
Then give a short numbered solution (1-3 lines) showing the key equations and arithmetic.
Do NOT provide private chain-of-thought — only the final answer and concise solution steps.

Example:
Problem: "Anna is 6 years older than Ben. In 4 years Anna will be twice Ben's age. How old are they now?"
A: Answer: Anna = 14, Ben = 8
1) Let Ben = x, Anna = x + 6. In 4 years: (x+6)+4 = 2*(x+4).
2) Solve: x+10 = 2x+8 → x = 2 → Ben = 2, Anna = 8. (adjusted for arithmetic)
(Above is an illustrative formatting example — keep steps short.)

Example:
Problem: "Mary is 3 times as old as Sam. In 5 years, Mary will be 15 and Sam 5. What are their current ages?"
A: Answer: Mary = 10, Sam = 5
1) In 5 years Mary = 15, Sam = 10 → current ages subtract 5 → Mary 10, Sam 5.
2) Check: Mary = 2 * Sam now (10 vs 5) — works.

Now solve:
Problem: "John is twice as old as Sara. In 6 years, the sum of their ages will be 42. How old are they now?"

""",

    # 2) Logical validity analysis - final verdict + concise justification
    "logical_validity": """
Decide whether each argument is logically valid. For each:
A: Begin with "Answer: Valid" or "Answer: Not valid"
Then provide 1-3 brief numbered reasons (contrapositive, counterexample, or short formal reasoning).
Keep explanations concise — do NOT give chain-of-thought.

Example:
Argument: "All dogs bark. Rex is a dog. Therefore, Rex barks."
A: Answer: Valid
1) This follows by universal instantiation: if all dogs have property P, any particular dog has P.
2) No counterexample exists inside premises.

Example:
Argument: "If it rains, the ground is wet. The ground is wet; therefore, it rained."
A: Answer: Not valid
1) This commits affirming the consequent — wet ground could have other causes (sprinkler).
2) Provide counterexample: ground wet due to sprinkler even though it didn't rain.

Now analyze:
Statement: "All musicians can read sheet music. Alex cannot read sheet music. Therefore, Alex is not a musician."

""",

    # 3) Plot hole detection - yes/no, evidence, and brief fix
    "plot_hole": """
Determine whether the story contains a plot hole. For each story:
A: Start with "Answer: Plot hole: Yes" or "Answer: Plot hole: No"
Then provide 2 concise numbered evidences (specific sentences/events) that support the verdict.
If Yes, add one 1-2 sentence suggested fix.
No private chain-of-thought — only the compact analysis.

Example:
Story: "A detective found a locked room with no windows. The suspect has a solid alibi downtown at the time of the murder."
A: Answer: Plot hole: Yes
1) Evidence: No explanation for how the suspect could be both downtown and in the locked room.
2) Evidence: The timeline given in the story is inconsistent with travel times.
Fix: Give the suspect a believable transportation method (e.g., had an accomplice) or tighten the alibi timeline.

Example:
Story: "Mia forgot her passport at home but still boarded the international flight without showing ID."
A: Answer: Plot hole: Yes
1) Evidence: Airports require ID; no checkpoint described.
2) Evidence: No mechanism in story explains bypassing security.
Fix: Add a scene where Mia obtains emergency travel papers from the consulate.

Now analyze:
Story: "Iqra was excited for her big piano recital. She practiced every day after school, and her teacher told her she was ready. On the day of the performance, she arrived at the concert hall early. However, when she sat at the piano, she suddenly realized she had forgotten her sheet music at home and began to panic.

But then, she took a deep breath and remembered the entire piece perfectly. She played through it flawlessly from memory, and the audience applauded loudly when she finished.

After the performance, Iqra told her teacher that forgetting the sheet music didn't matter because she actually had a printed copy inside her bag the whole time.
"
""",

    # 4) Laptop recommendation under $500 - provide single top pick + concise justification
    "laptop_recommend": """
Recommend the best laptop for a student with a $500 budget. For each recommendation:
A: Begin with "Answer: <Model Name> — approx $XXX"
Then give 3 concise numbered reasons (one sentence each) focused on productivity, battery life, and durability.
Finally list 2 short alternative options (one line each: name + one pro).
Keep the justification short; do NOT provide internal chain-of-thought.
If prices vary, give approximate USD price.

Example:
Constraints: Budget $500
A: Answer: Lenovo IdeaPad 3 (example) — approx $449
1) CPU/RAM: Sufficient CPU and 8GB RAM for web-based study and documents.
2) Battery: Listed battery life ~8 hours suitable for classes.
3) Durability/portability: Lightweight chassis and good build for students.
Alternatives:
- Acer Aspire 5 — Good performance/price.
- HP Chromebook — Excellent battery life and low cost.

Now recommend:
Constraints: Budget $500
"""
}

for name, prompt in prompts.items():
    print("----\nPrompt key:", name)
    response = model.generate_content(prompt)
    # Safe extraction — avoid using response.text directly
    if getattr(response, "candidates", None) and response.candidates and getattr(response.candidates[0].content, "parts", None):
        print(response.candidates[0].content.parts[0].text)
    else:
        # If no parts, print the raw response object for debugging
        print("No full text part returned; raw response:\n", response)


----
Prompt key: math_age
A: Answer: John = 20, Sara = 10
1) Let Sara's age = S, John's age = J. So J = 2S.
2) In 6 years: (S+6) + (J+6) = 42. Substitute J=2S: (S+6) + (2S+6) = 42.
3) Solve: 3S + 12 = 42 → 3S = 30 → S = 10. John = 2 * 10 = 20.
----
Prompt key: logical_validity
Answer: Valid
1) This argument follows the valid logical form of Modus Tollens.
2) The conclusion is a necessary consequence if the premises are true.
3) No counterexample exists where the premises are true but the conclusion is false.
----
Prompt key: plot_hole
A: Answer: Plot hole: Yes
1) Evidence: The story states Iqra "suddenly realized she had forgotten her sheet music at home and began to panic," implying a factual understanding of the situation.
2) Evidence: After the performance, Iqra reveals she "actually had a printed copy inside her bag the whole time," which contradicts her earlier "realization" and subsequent panic.
Fix: Change "realized" to "thought" or "believed" to indicate her initial panic was b







## Write prompts for the above situations.
- A good prompt (clear, effective, uses right context/examples if needed)
- An average prompt (somewhat vague or lacking)
- A bad prompt (unclear, missing goal/context)
- Comment briefly on each (e.g., What’s missing? Why does it work? etc)

PS: you are allowed to use LLM of your choice using its api key.

### Deliverable
Please strcitly adhere to the submission guidleines, i.e.,  submit the .ipynb file with complete code.